In [1]:
import rioxarray as rxr
import xarray as xr
from pyproj import Transformer
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import glob
import numpy as np
from matplotlib.animation import FuncAnimation
from rasterio.enums import Resampling
import pandas as pd
import os
from pathlib import Path

In [2]:
root = Path(".").resolve()
project_root = root if (root / "Notebooks").exists() else root.parent
out_dir = project_root / "Subglacial_Lakes_Data"

df = pd.read_csv(out_dir/"lake_and_null_loc.csv")
df

FileNotFoundError: [Errno 2] No such file or directory: '/Users/sofiasuhinin/Desktop/MLGEO2026_Subglacial_Lakes/Subglacial_Lakes_Data/lake_and_null_loc.csv'

In [5]:
remaMosaicPath = r"/Users/Ashley Howard/ESS569/rema_mosaic_100m_v2.0_dem.tif"
cryosatPath = r"/Users/Ashley Howard/ESS569/MLGEO2026_Subglacial_Lakes/Notebooks/cryosat_downloads/CS_OFFL_THEM_GRID__ANTARCTIC_" 

In [6]:
years = np.arange(2011, 2020) 
yearsString = [str(year) for year in years]
months = np.arange(1, 13)
monthsString = [str(month).zfill(2) for month in months]
allTime = ['2010_08', '2010_09', '2010_10', '2010_11', '2010_12']+[year + "_" + month for year in yearsString for month in monthsString]

In [7]:
rema = rxr.open_rasterio(remaMosaicPath, masked=True).squeeze()

In [8]:

def monthlyAnomaly(rema, date, long, lat):
    # Import CryoSat Data
    file_path = cryosatPath + date + "_V201.nc"

    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return None   # ← exit function safely

    with xr.open_dataset(file_path) as ds:
        elevation = ds['elevation'].transpose('time', 'y', 'x')
        # Find Center of the Lake
        transformer = Transformer.from_crs(
            "EPSG:4326",   # lat/lon
            "EPSG:3031",   # Antarctic Polar Stereo
            always_xy=True
        )

        x_center, y_center = transformer.transform(long, lat)

        # Define Bounding Box Around Center of the Lake
        half_size = 25_000  # 25 (edit)

        xmin = x_center - half_size
        xmax = x_center + half_size
        ymin = y_center - half_size
        ymax = y_center + half_size

    # Pull Boxes for CryoSat & REMA
        cryosat_small = elevation.sel(x=slice(xmin, xmax), y=slice(ymin, ymax))
        rema_small = rema.rio.clip_box(xmin,ymin,xmax,ymax)

        # Calculate Anomaly in Cryosat Based on REMA
        cryosat_small = cryosat_small.rio.write_crs("EPSG:3031")
        cryosat_small = cryosat_small.rio.set_spatial_dims(x_dim="x", y_dim="y")

        # Regrid Cryosat (2km) to match REMA (100 m) using nearest-neighbor
        cryosat_small_r = cryosat_small.rio.reproject_match(rema_small, resampling=Resampling.nearest)

            # commented out bc no longer projecting to cryosat
    #rema_small = rema_small.rio.set_spatial_dims(x_dim="x", y_dim="y")
    #rema_matched = rema_small.rio.reproject_match(cryosat_small_r.isel(time=0)) 

        anomaly = cryosat_small_r.isel(time=0) - rema_small

    return anomaly

In [15]:
from __future__ import annotations

import os
import errno
import time
import tempfile
from pathlib import Path
from typing import Iterable

import xarray as xr

def locationAnomaly(longitude: float,
                    latitude: float,
                    file_path: str,
                    all_time: Iterable,
                    *,
                    output_dir: str = "lakesTS",
                    retries: int = 5,
                    backoff: float = 0.5) -> Path:
    """
    Build a time-stacked anomaly dataset and write to NetCDF with
    Windows-friendly, atomic & retrying write semantics.

    Parameters
    ----------
    longitude : float
    latitude : float
    file_path : str
        Base file name (without extension).
    all_time : iterable
        Iterable of datetime-like values passed to monthlyAnomaly(...).
    output_dir : str
        Target directory for output NetCDF.
    retries : int
        Max attempts for handling PermissionError / EACCES.
    backoff : float
        Initial backoff in seconds; doubles each retry.

    Returns
    -------
    Path
        The final path to the written NetCDF.
    """
    # 1) Compute the stacked anomalies first (fail fast before file ops)
    dataarrays = []
    for date in all_time:
        da = monthlyAnomaly(rema, date, longitude, latitude)  # assumes these exist in your environment
        dataarrays.append(da)
    dataarrays = [da for da in dataarrays if da is not None]
    stacked = xr.concat(dataarrays, dim="time")
    ds = xr.Dataset({"anomaly": stacked})

    # 2) Ensure output directory exists (handle permissions here too)
    out_dir = Path(output_dir)
    try:
        out_dir.mkdir(parents=True, exist_ok=True)
    except PermissionError as e:
        raise PermissionError(
            f"Cannot create or access directory: {out_dir}. "
            f"Check write permissions or try a different location."
        ) from e

    # 3) Build a sanitized destination path
    #    (Windows doesn't like certain characters in filenames)
    safe_name = _sanitize_filename(file_path)
    dest_path = out_dir / f"{safe_name}.nc"

    # 4) Atomic write: write to a closed temp file, then os.replace()
    #    (on Windows, NamedTemporaryFile must be closed before reuse)
    attempt = 0
    tmp_path: Path | None = None
    try:
        while True:
            try:
                with tempfile.NamedTemporaryFile(
                    mode="wb",
                    suffix=".nc",
                    prefix=f".{safe_name}.tmp.",
                    dir=str(out_dir),
                    delete=False  # we will clean up explicitly
                ) as tmp:
                    tmp_path = Path(tmp.name)

                # Use a separate handle to ensure xarray closes properly before replace
                # If your arrays are dask-backed, consider compute() before write for predictability:
                # ds = ds.compute()
                # Use a lock-free write; xarray handles locking internally for netCDF4/h5netcdf engines
                ds.to_netcdf(tmp_path, mode="w")

                # Force data flush & close before replace
                ds.close()

                # Atomic move into final place
                os.replace(tmp_path, dest_path)
                return dest_path

            except PermissionError as e:
                # Windows: file in use, AV scanning, or no rights
                attempt += 1
                if attempt > retries:
                    _raise_with_context(e, dest_path, attempt)
                # Exponential backoff
                time.sleep(backoff)
                backoff *= 2
                continue

            except OSError as e:
                # Treat EACCES like PermissionError as well
                if e.errno in (errno.EACCES, errno.EPERM):
                    attempt += 1
                    if attempt > retries:
                        _raise_with_context(e, dest_path, attempt)
                    time.sleep(backoff)
                    backoff *= 2
                    continue
                # Other OS errors: bubble up
                raise

    finally:
        # Best-effort cleanup of temp file if it still exists
        try:
            if tmp_path and tmp_path.exists():
                tmp_path.unlink(missing_ok=True)
        except Exception:
            # Don't mask the original exception
            pass
        # Ensure dataset is closed even if an exception occurred
        try:
            ds.close()
        except Exception:
            pass


def _sanitize_filename(name: str) -> str:
    """
    Remove characters problematic on Windows: \\ / : * ? \" < > | and control chars.
    Also trims spaces & dots at the end which Windows disallows for names.
    """
    bad = '<>:"/\\|?*'
    cleaned = "".join(ch for ch in name if ch not in bad and ord(ch) >= 32)
    cleaned = cleaned.strip().rstrip(".")
    return cleaned or "output"


def _raise_with_context(e: BaseException, path: Path, attempts: int) -> None:
    raise PermissionError(
        "Failed to write NetCDF after multiple attempts.\n"
        f"Target: {path}\n"
        f"Attempts: {attempts}\n\n"
        "Possible causes on Windows:\n"
        "  • The file is open in another app (e.g., Excel, Panoply, Explorer preview).\n"
        "  • Antivirus or backup tools are temporarily locking the file.\n"
        "  • Insufficient permissions to the folder.\n"
        "  • Corporate policies (e.g., Controlled Folder Access) blocking writes.\n\n"
        "Try:\n"
        "  • Close any app that might be using the file.\n"
        "  • Write to a different directory (e.g., your user Documents folder).\n"
        "  • Disable the Explorer preview pane.\n"
        "  • Temporarily pause AV scanning on the folder (if allowed).\n"
        "  • Run Python as a user with write permissions.\n"
    ) from e


In [16]:
# Loop through rows
for _, row in df.iterrows():
    locationAnomaly(
        row["lon"],
        row["lat"],
        row["name"],
        allTime
    )

File not found: /Users/Ashley Howard/ESS569/MLGEO2026_Subglacial_Lakes/Notebooks/cryosat_downloads/CS_OFFL_THEM_GRID__ANTARCTIC_2015_01_V201.nc
File not found: /Users/Ashley Howard/ESS569/MLGEO2026_Subglacial_Lakes/Notebooks/cryosat_downloads/CS_OFFL_THEM_GRID__ANTARCTIC_2015_07_V201.nc
File not found: /Users/Ashley Howard/ESS569/MLGEO2026_Subglacial_Lakes/Notebooks/cryosat_downloads/CS_OFFL_THEM_GRID__ANTARCTIC_2015_01_V201.nc
File not found: /Users/Ashley Howard/ESS569/MLGEO2026_Subglacial_Lakes/Notebooks/cryosat_downloads/CS_OFFL_THEM_GRID__ANTARCTIC_2015_07_V201.nc
File not found: /Users/Ashley Howard/ESS569/MLGEO2026_Subglacial_Lakes/Notebooks/cryosat_downloads/CS_OFFL_THEM_GRID__ANTARCTIC_2015_01_V201.nc
File not found: /Users/Ashley Howard/ESS569/MLGEO2026_Subglacial_Lakes/Notebooks/cryosat_downloads/CS_OFFL_THEM_GRID__ANTARCTIC_2015_07_V201.nc
File not found: /Users/Ashley Howard/ESS569/MLGEO2026_Subglacial_Lakes/Notebooks/cryosat_downloads/CS_OFFL_THEM_GRID__ANTARCTIC_2015_01_

In [17]:
import os
import glob
import numpy as np
from sklearn.model_selection import train_test_split

# Folder containing netCDF files
data_dir = r"/Users/Ashley Howard/ESS569/MLGEO2026_Subglacial_Lakes/Notebooks/lakesTS"

# Get all .nc files
files = sorted(glob.glob(os.path.join(data_dir, "*.nc")))

# Assign block IDs
block_ids = np.arange(len(files))

# First split: 70% train, 30% temp
train_files, temp_files = train_test_split(
    files,
    test_size=0.30,
    random_state=42,   # ensures reproducibility
    shuffle=True
)

# Second split: split the 30% into 15% val, 15% test
val_files, test_files = train_test_split(
    temp_files,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print(f"Total files: {len(files)}")
print(f"Train: {len(train_files)}")
print(f"Validation: {len(val_files)}")
print(f"Test: {len(test_files)}")

Total files: 105
Train: 73
Validation: 16
Test: 16


In [18]:
import pandas as pd

split_labels = (
    ["train"] * len(train_files) +
    ["val"] * len(val_files) +
    ["test"] * len(test_files)
)

all_split_files = np.concatenate([train_files, val_files, test_files])

df = pd.DataFrame({
    "file": all_split_files,
    "dataset": split_labels
})

df.to_csv(out_dir/"dataset_split.csv", index=False)